# plaude-local - multilingual regression suite on a Colab GPU

Runs `tests/regression/run_regression.py` (both the Python and the PowerShell
tool, transcribe + translate to English) on a cloud GPU with the best models:
**Whisper large-v3** and **translategemma:12b** (via Ollama).

This is the **default way to run the suite** - no local GPU needed.

1. **Runtime -> Change runtime type -> GPU** (T4 is enough), then **Runtime -> Run all**.
2. Results go to Google Drive (`MyDrive/plaude-regression/`), saved after every run.
3. Keep this tab open (Colab may stop a session with no open tab). If your
   computer or browser crashes, just reopen this notebook - the run continues in the
   cloud; do **not** press Run all while it is still running. If Colab ended the
   session, **Run all** again: passed runs are kept, failed ones run again.
4. When it's done, download `baseline.json` + `results.json` from that Drive folder
   and copy `baseline.json` into `tests/regression/` in your local repo.

Timing on a T4: the default 24-item sample is 96 runs, about 5-6 hours. Colab's free
tier can stop long sessions; set `IMPL = "python"` for a ~half-length first pass.

## 1. Settings

In [ ]:
SAMPLE = 24          # stratified sample size (0 = the full 300-item / 1200-run suite)
IMPL = "both"        # "both", "python" or "powershell"
BRANCH = "claude/local-speech-to-text-tool-ancn9e"   # the branch this notebook was opened from
REPO = "thewongdirection/my-plaude"
RUN_NAME = f"sample{SAMPLE}" if SAMPLE else "full"

## 2. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 3. Keep results on Google Drive (survives disconnects)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
OUT = f"/content/drive/MyDrive/plaude-regression/{RUN_NAME}"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)

## 4. Get the code and the corpus

A public repo clones directly. If the repo is **private**, add a GitHub token with
read access as a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar)
- never paste it into a cell.

In [ ]:
import base64, os, subprocess
url = f"https://github.com/{REPO}.git"
env = dict(os.environ)
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    # Pass the token as an HTTP header through git's environment config: it never
    # appears in a command line, an error message or .git/config.
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
except Exception:
    pass  # no secret: public clone
def git(*args):
    r = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f"git {args[0]} failed: {r.stderr.strip()[-500:]}")
if not os.path.isdir("/content/my-plaude"):
    git("clone", "--depth", "1", "-b", BRANCH, url, "/content/my-plaude")
else:
    git("-C", "/content/my-plaude", "pull", "--ff-only")
%cd /content/my-plaude
!git log --oneline -1 && du -sh tests/regression/corpus

## 5. Install the tools (FFmpeg, faster-whisper, whisper-ctranslate2, CUDA libraries)

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
!pip -q install faster-whisper whisper-ctranslate2 zhconv nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
# CTranslate2 loads cuBLAS / cuDNN by name: put the pip-installed CUDA libraries on the
# library path for every tool the runner starts.
import glob, os, site
libs = sorted({p for sp in site.getsitepackages() for p in glob.glob(f"{sp}/nvidia/*/lib")})
os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
print("\n".join(libs))

## 6. Install PowerShell 7 (`pwsh`) for the PowerShell tool

In [ ]:
import shutil
if not shutil.which("pwsh"):
    !wget -q https://github.com/PowerShell/PowerShell/releases/download/v7.4.6/powershell-7.4.6-linux-x64.tar.gz -O /tmp/pwsh.tgz
    !mkdir -p /opt/microsoft/powershell/7 && tar zxf /tmp/pwsh.tgz -C /opt/microsoft/powershell/7
    !chmod +x /opt/microsoft/powershell/7/pwsh && ln -sf /opt/microsoft/powershell/7/pwsh /usr/local/bin/pwsh
import subprocess  # no shell: "$PSVersionTable" must reach pwsh unexpanded
print("pwsh", subprocess.run(["pwsh", "-NoProfile", "-Command", "$PSVersionTable.PSVersion.ToString()"],
                             capture_output=True, text=True).stdout.strip())

## 7. Install Ollama and the translation model (translategemma:12b, ~8 GB)

(The 12B is the best TranslateGemma that fits a 16 GB T4 next to Whisper large-v3;
the runner picks it automatically via `--translate-model best`.)

`pciutils` lets the installer detect the GPU. The model is kept loaded
(`OLLAMA_KEEP_ALIVE=-1`): by default Ollama unloads it after 5 idle minutes, and
reloading 8 GB mid-run made translations time out. The last lines load the model
and STOP if it is not on the GPU.

In [ ]:
import json, os, shutil, subprocess, time, urllib.request
if not shutil.which("ollama"):
    !apt-get -qq install -y pciutils zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
env = dict(os.environ, OLLAMA_KEEP_ALIVE="-1")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"),
                 stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
!ollama pull translategemma:12b
# Warm-up: load the model now (kept loaded for the whole run) and check it is on the GPU.
req = urllib.request.Request("http://127.0.0.1:11434/api/generate", method="POST",
    headers={"Content-Type": "application/json"},
    data=json.dumps({"model": "translategemma:12b", "prompt": "Translate to English: Hola",
                     "stream": False, "keep_alive": -1}).encode())
print("warm-up reply:", json.loads(urllib.request.urlopen(req, timeout=900).read())["response"][:80])
ps = json.loads(urllib.request.urlopen("http://127.0.0.1:11434/api/ps").read())["models"]
vram = {m["name"]: m.get("size_vram", 0) for m in ps}
print("loaded (bytes in GPU memory):", vram)
assert vram.get("translategemma:12b", 0) > 0, (
    "Ollama is running translategemma:12b on the CPU - translations would time out. "
    "Check /content/ollama.log, then Runtime -> Disconnect and delete runtime and Run all.")
print("OK: translategemma:12b is on the GPU")

## 8. Prerequisite checks (both tools)

In [ ]:
!python -m plaude_local --check
!pwsh -NoProfile -File powershell/Plaude-Local.ps1 -Check

## 9. Run the regression suite

Output streams below and to `run.log` on Drive; `results.json` is updated after every
run. Re-running this cell after a disconnect resumes where it stopped (passed runs are
kept, failed ones run again). Each tool gets a 15-minute LLM timeout, a run killed by
the system is retried once, and if anything still failed, a second pass retries just
those runs.

In [ ]:
import subprocess, sys
cmd = [sys.executable, "-u", "tests/regression/run_regression.py", "--impl", IMPL,
       "--out", OUT, "--resume", "--update-baseline"]
if SAMPLE:
    cmd += ["--sample", str(SAMPLE)]
for attempt in (1, 2):  # a second pass re-runs only the runs that failed (--resume)
    with open(f"{OUT}/run.log", "a", encoding="utf-8") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in proc.stdout:
                print(line, end="")
                log.write(line); log.flush()
        except KeyboardInterrupt:  # Runtime -> Interrupt: stop the runner (it kills its tools)
            proc.terminate(); proc.wait()
            raise
    code = proc.wait()
    print(f"pass {attempt} exit code:", code)
    if code == 0:
        break

## 10. Save the baseline next to the results

In [ ]:
import shutil
shutil.copy("tests/regression/baseline.json", f"{OUT}/baseline.json")
!ls -la "$OUT"
print("Download baseline.json and results.json from Google Drive:", OUT)